# Fall model training - ST-GCN on YOLO11-pose skeletons

Run every cell top to bottom (or **Save Version -> Save & Run All**).

Before running, in the right-hand panel:
- **Accelerator: GPU T4 x2**
- **Internet: on**
- **Add Input: "Le2i Fall Dataset"** (optional: your own `yolo11n-pose.pt` uploaded as a private dataset)

What happens:
1. find the videos and their fall labels
2. turn every video into skeletons with **yolo11n-pose** (used as-is, not trained)
3. cut the skeletons into 3-second windows, labelled fall / not fall, split by video
4. train **ST-GCN** - the only model trained here
5. test it on videos it never saw

Result: **`/kaggle/working/fall_stgcn.pt`** - download it from the **Output** tab and put it in the project's `models/` folder.

In [ ]:
!pip install -q ultralytics==8.4.136 imageio-ffmpeg

In [ ]:
# --- 1. Settings, and find the videos and their labels ----------------------
import os, re, json, random, time
from pathlib import Path

import cv2
import numpy as np
import torch
from torch import nn

INPUT_DIR = Path(os.environ.get("INPUT_DIR", "/kaggle/input"))
WORK_DIR = Path(os.environ.get("WORK_DIR", "/kaggle/working"))
WORK_DIR.mkdir(parents=True, exist_ok=True)

# These must match the live app, or the model learns from a different picture
# than it will see: frames scaled down to 640 wide, sampled at 15 fps, yolo11n-pose.
WIDTH = 640
RATE = 15.0
WINDOW = 45        # 3 seconds at 15 fps - how much the model looks at, at once
STRIDE = 4         # a new window every 4 frames
CONF_MIN = 0.3     # a keypoint below this confidence counts as not seen
SEED = 0
EPOCHS = int(os.environ.get("EPOCHS", "40"))

VIDEO_EXTS = {".avi", ".mp4", ".mov", ".mkv"}


def find_annotation(video):
    """Le2i keeps one .txt per video, same name, in a folder beside Videos/
    (Annotation_files/)."""
    for folder in video.parent.parent.iterdir():
        if folder.is_dir() and "annot" in folder.name.lower():
            candidate = folder / (video.stem + ".txt")
            if candidate.exists():
                return candidate
    beside = video.with_suffix(".txt")
    return beside if beside.exists() else None


def read_fall(txt):
    """The first two lines are the frame the fall starts and the frame it ends.
    Returns (start, end), "none" for a video without a fall (0 and 0), or
    "unknown" if the file doesn't look like that."""
    numbers = []
    for line in txt.read_text(errors="ignore").splitlines()[:2]:
        found = re.findall(r"\d+", line)
        if len(found) != 1:
            return "unknown"
        numbers.append(int(found[0]))
    if len(numbers) < 2:
        return "unknown"
    start, end = numbers
    return (start, end) if end > start > 0 else "none"


items, seen_before = [], set()
left_out = {"no label file": 0, "label unreadable": 0, "duplicate copy": 0}
for video in sorted(p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in VIDEO_EXTS):
    key = (video.stem.lower(), video.stat().st_size)
    if key in seen_before:
        left_out["duplicate copy"] += 1        # the same video twice would leak into the test
        continue
    seen_before.add(key)
    annotation = find_annotation(video)
    if annotation is None:
        left_out["no label file"] += 1
        continue
    fall = read_fall(annotation)
    if fall == "unknown":
        left_out["label unreadable"] += 1
        continue
    items.append({"id": re.sub(r"[^A-Za-z0-9]+", "_", str(video.relative_to(INPUT_DIR))),
                  "video": video, "annotation": annotation, "fall": fall})

with_fall = sum(1 for it in items if it["fall"] != "none")
print(f"videos with labels: {len(items)}  (with a fall: {with_fall}, without: {len(items) - with_fall})")
print("left out:", left_out)
example = next((it for it in items if it["fall"] != "none"), None)
if example:
    print(f"\nexample label file: {example['annotation'].relative_to(INPUT_DIR)}")
    print("  first lines:", example["annotation"].read_text(errors="ignore").splitlines()[:3])
    print("  read as: fall from frame", example["fall"][0], "to", example["fall"][1])
if not items or with_fall == 0 or with_fall == len(items):
    print("\nWARNING: need videos both with and without a fall. Stop here and check the dataset.")

In [ ]:
# --- 2. Turn every video into skeletons (yolo11n-pose, used as-is) ----------
# This runs in a separate worker process. Reading a Le2i video killed the whole
# notebook the first time (some files have a broken audio track), so now: the
# video is read without its audio at all, and if something still crashes, only
# the worker dies - the notebook notes which video, skips it and carries on.
import subprocess, sys

WORKER = r'''
import faulthandler, json, os, sys
faulthandler.enable()                      # if it crashes, print where
from pathlib import Path
import cv2
import imageio_ffmpeg
import numpy as np
from ultralytics import YOLO

tasks_file, progress_file, device, weights = sys.argv[1:5]
width, rate = int(sys.argv[5]), float(sys.argv[6])
pose_model = YOLO(weights)


def main_person(result):
    """The biggest person in the frame, as 17 x (x, y, confidence). Zeros if nobody."""
    if result.keypoints is None or len(result.boxes) == 0:
        return np.zeros((17, 3), np.float32)
    boxes = result.boxes.xyxy.cpu().numpy()
    biggest = int(((boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])).argmax())
    xy = result.keypoints.xy[biggest].cpu().numpy()
    conf = result.keypoints.conf
    conf = conf[biggest].cpu().numpy() if conf is not None else np.ones(17, np.float32)
    return np.concatenate([xy, conf[:, None]], axis=1).astype(np.float32)


def extract(video):
    """Samples the video at `rate` the way the live app paces the camera. Returns
    the skeletons and each sample's frame number (1-based, like Le2i's labels).
    Only the picture is decoded - the audio track is never touched."""
    frames = imageio_ffmpeg.read_frames(video, pix_fmt="bgr24", output_params=["-map", "0:v:0"])
    meta = next(frames)
    fps = meta.get("fps") or 25.0
    w, h = meta["size"]
    skeletons, numbers, batch = [], [], []
    n, next_t = 0, 0.0

    def run_batch():
        if batch:
            skeletons.extend(main_person(r) for r in pose_model(batch, verbose=False, device=device))
            batch.clear()

    for raw in frames:
        t = n / fps
        n += 1
        if t + 1e-9 < next_t:
            continue
        next_t += 1 / rate
        frame = np.frombuffer(raw, np.uint8).reshape(h, w, 3).copy()
        if w > width:
            frame = cv2.resize(frame, (width, round(h * width / w)), interpolation=cv2.INTER_AREA)
        batch.append(frame)
        numbers.append(n)
        if len(batch) == 32:
            run_batch()
    run_batch()
    kp = np.stack(skeletons) if skeletons else np.zeros((0, 17, 3), np.float32)
    return kp, np.array(numbers)


for k, (video, out) in enumerate(json.load(open(tasks_file))):
    Path(progress_file).write_text(str(k))
    try:
        kp, numbers = extract(video)
    except Exception as error:             # unreadable video: report it, carry on
        print("failed", k, repr(error)[:300], flush=True)
        continue
    part = out[:-len(".npz")] + ".part.npz"
    np.savez_compressed(part, kp=kp, numbers=numbers)
    os.replace(part, out)                  # never leaves a half-written file behind
    print("done", k, flush=True)
'''

CACHE = WORK_DIR / "keypoints"
CACHE.mkdir(exist_ok=True)
WORKER_PATH = WORK_DIR / "extract_worker.py"
WORKER_PATH.write_text(WORKER)
uploaded = sorted(INPUT_DIR.rglob("yolo11n-pose.pt"))
WEIGHTS = str(uploaded[0]) if uploaded else "yolo11n-pose.pt"   # your copy, else the official one
device = os.environ.get("POSE_DEVICE") or ("0" if torch.cuda.is_available() else "cpu")
print("pose model:", uploaded[0] if uploaded else "official download", "  device:", device)

counter = {"done": 0}
started = time.time()


def out_path(item):
    return CACHE / (item["id"] + ".npz")


def run_worker(batch, on):
    """Runs the worker over `batch` on device `on`. Returns (the index it crashed
    at, or None), [(index, reason)] for videos it couldn't read, its last error lines."""
    tasks, progress, errors = WORK_DIR / "tasks.json", WORK_DIR / "progress.txt", WORK_DIR / "worker_errors.txt"
    tasks.write_text(json.dumps([(str(it["video"]), str(out_path(it))) for it in batch]))
    progress.unlink(missing_ok=True)
    unreadable = []
    with open(errors, "w") as err:
        proc = subprocess.Popen(
            [sys.executable, str(WORKER_PATH), str(tasks), str(progress), on, WEIGHTS, str(WIDTH), str(RATE)],
            stdout=subprocess.PIPE, stderr=err, text=True, cwd=WORK_DIR)
        for line in proc.stdout:
            word, _, rest = line.strip().partition(" ")
            if word == "done":
                counter["done"] += 1
                if counter["done"] % 10 == 0:
                    print(f"{counter['done']}/{len(items)} videos  {time.time() - started:.0f}s", flush=True)
            elif word == "failed":
                k, _, why = rest.partition(" ")
                unreadable.append((int(k), why))
        proc.wait()
    crashed = None
    if proc.returncode != 0:
        crashed = int(progress.read_text()) if progress.exists() else 0
    return crashed, unreadable, errors.read_text(errors="ignore").strip().splitlines()[-12:]


left_out_videos = {}
pending = [it for it in items if not out_path(it).exists()]
counter["done"] = len(items) - len(pending)
crashes_in_a_row = 0
while pending:
    crashed, unreadable, last_lines = run_worker(pending, device)
    for k, why in unreadable:
        left_out_videos[pending[k]["id"]] = why
    if crashed is None:
        break
    if crashed > 0:
        crashes_in_a_row = 0                  # it got through some videos first
    video = pending[crashed]
    print(f"\nthe worker crashed on {video['video'].name} (device {device}). Its last messages:")
    print("  " + "\n  ".join(last_lines))
    if device != "cpu":
        print("trying that video on the CPU, to tell a GPU problem from a bad video...")
        again, unreadable_again, _ = run_worker([video], "cpu")
        if again is None and not unreadable_again:
            device = "cpu"
            print("it worked on the CPU, so the GPU was the problem - carrying on on the CPU (slower)\n")
            crashes_in_a_row = 0
            pending = pending[crashed + 1:]
            continue
    left_out_videos[video["id"]] = "crashed the worker"
    crashes_in_a_row += 1
    if crashes_in_a_row >= 5:
        raise RuntimeError("5 videos in a row crashed - stopping here. Send the messages above.")
    pending = pending[crashed + 1:]

if left_out_videos:
    print(f"\nleft out {len(left_out_videos)} videos that could not be read:")
    for name, why in list(left_out_videos.items())[:10]:
        print("  ", name, "-", why[:150])
items = [it for it in items if it["id"] not in left_out_videos and out_path(it).exists()]
for it in items:
    saved = np.load(out_path(it))
    it["kp"], it["numbers"] = saved["kp"], saved["numbers"]
print(f"skeletons ready for {len(items)} videos  ({time.time() - started:.0f}s)")

In [ ]:
# --- 3. Cut into 3-second windows, labelled, split by video -----------------
def normalise(window):
    """Pixels -> centred on the hips and scaled to body size, so where someone
    stands and how far they are from the camera don't matter. One centre for the
    whole window, not one per frame, so the hips dropping in a fall is kept.
    The live app must do exactly this before asking the model."""
    xy = window[..., :2].copy()
    conf = window[..., 2].copy()
    seen = conf >= CONF_MIN
    hips = seen[:, 11] & seen[:, 12]
    if hips.any():
        centre = np.median((xy[hips, 11] + xy[hips, 12]) / 2, axis=0)
    elif seen.any():
        centre = xy[seen].mean(axis=0)
    else:
        return None
    size = 0.0
    for t in range(len(window)):
        points = xy[t][seen[t]]
        if len(points) >= 4:
            size = max(size, float((points.max(axis=0) - points.min(axis=0)).max()))
    if size < 1:
        return None
    xy = (xy - centre) / size
    xy[~seen] = 0
    conf[~seen] = 0
    return np.concatenate([xy, conf[..., None]], axis=-1).astype(np.float32)


def fall_span(item):
    """The fall's first and last sample index, or None."""
    if item["fall"] == "none":
        return None
    numbers = item["numbers"]
    first = int(np.searchsorted(numbers, item["fall"][0]))
    last = int(np.searchsorted(numbers, item["fall"][1], side="right")) - 1
    return (first, last) if last >= first else None


def windows(item):
    """(start, sample, label) for every window of one video. label 1: the whole
    fall happens inside it. 0: nowhere near a fall. None: it catches only part of
    the fall - neither one nor the other, so it is left out of training."""
    kp, span = item["kp"], fall_span(item)
    for start in range(0, len(kp) - WINDOW + 1, STRIDE):
        end = start + WINDOW
        visible = ((kp[start:end, :, 2] >= CONF_MIN).sum(axis=1) >= 5).mean()
        if visible < 0.5:
            continue                      # the person is missing for most of it
        sample = normalise(kp[start:end])
        if sample is None:
            continue
        if span is None:
            label = 0
        else:
            first, last = span
            if start <= first and last < end:
                label = 1
            elif last - first + 1 > WINDOW and start <= (first + last) // 2 < end:
                label = 1                 # a fall longer than 3 seconds: its middle is enough
            elif end <= first - 8 or start > last + 8:
                label = 0
            else:
                label = None
        yield start, sample, label


# Split by VIDEO, never by window: windows of one video in both training and
# testing would make the results look far better than they are.
rng = random.Random(SEED)
for group in ([it for it in items if it["fall"] != "none"], [it for it in items if it["fall"] == "none"]):
    rng.shuffle(group)
    a, b = int(len(group) * 0.70), int(len(group) * 0.85)
    for k, it in enumerate(group):
        it["split"] = "train" if k < a else ("val" if k < b else "test")

X = {s: [] for s in ("train", "val", "test")}
Y = {s: [] for s in ("train", "val", "test")}
for it in items:
    for _, sample, label in windows(it):
        if label is not None:
            X[it["split"]].append(sample)
            Y[it["split"]].append(label)
for s in X:
    X[s] = np.stack(X[s]) if X[s] else np.zeros((0, WINDOW, 17, 3), np.float32)
    Y[s] = np.array(Y[s], dtype=np.int64)
    count = sum(1 for it in items if it["split"] == s)
    print(f"{s:5}: {count:3} videos, {len(Y[s]):6} windows, {int(Y[s].sum()):5} of them falls")

In [ ]:
# --- 4. Train ST-GCN - the only model trained here --------------------------
EDGES = [(0, 1), (0, 2), (1, 3), (2, 4), (0, 5), (0, 6), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
         (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]   # COCO-17, as YOLO gives them
MIRROR = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]          # left <-> right


def skeleton_graph():
    a = np.eye(17, dtype=np.float32)
    for i, j in EDGES:
        a[i, j] = a[j, i] = 1
    d = a.sum(axis=1)
    return a / np.sqrt(np.outer(d, d))


class Block(nn.Module):
    """One ST-GCN layer: mix each joint with its neighbours, then look across time."""

    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.graph = nn.Parameter(torch.tensor(skeleton_graph()))   # learnable, starts as the skeleton
        self.spatial = nn.Conv2d(c_in, c_out, 1)
        self.temporal = nn.Sequential(
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, (9, 1), (stride, 1), (4, 0)),
            nn.BatchNorm2d(c_out), nn.Dropout(0.2))
        self.shortcut = (nn.Identity() if c_in == c_out and stride == 1 else
                         nn.Sequential(nn.Conv2d(c_in, c_out, 1, (stride, 1)), nn.BatchNorm2d(c_out)))
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):                                  # batch, channels, time, joints
        y = torch.einsum("nctv,vw->nctw", self.spatial(x), self.graph)
        return self.relu(self.temporal(y) + self.shortcut(x))


class FallSTGCN(nn.Module):
    """Input: batch x 45 frames x 17 joints x (x, y, confidence), from normalise().
    Output: two scores, [not fall, fall]."""

    def __init__(self):
        super().__init__()
        self.input_norm = nn.BatchNorm1d(17 * 3)
        self.blocks = nn.Sequential(
            Block(3, 64), Block(64, 64), Block(64, 128, stride=2),
            Block(128, 128), Block(128, 256, stride=2), Block(256, 256))
        self.head = nn.Linear(256, 2)

    def forward(self, x):                                  # batch, time, joints, 3
        n, t, v, c = x.shape
        x = self.input_norm(x.permute(0, 2, 3, 1).reshape(n, v * c, t))
        x = x.reshape(n, v, c, t).permute(0, 2, 3, 1)      # batch, 3, time, joints
        return self.head(self.blocks(x).mean(dim=(2, 3)))


def augment(x):
    """Random mirror, a small turn and resize, a little jitter - more variety
    from the same clips. Joints that weren't seen stay at zero."""
    x = x.clone()
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip[:, None, None, None], x[:, :, MIRROR], x)
    x[..., 0] = x[..., 0] * torch.where(flip, -1.0, 1.0)[:, None, None]
    angle = (torch.rand(n, device=x.device) * 2 - 1) * (10 * np.pi / 180)
    scale = 0.9 + 0.2 * torch.rand(n, device=x.device)
    cos = (torch.cos(angle) * scale)[:, None, None]
    sin = (torch.sin(angle) * scale)[:, None, None]
    px, py = x[..., 0].clone(), x[..., 1].clone()
    seen = (x[..., 2] > 0).float()
    x[..., 0] = (cos * px - sin * py + 0.01 * torch.randn_like(px)) * seen
    x[..., 1] = (sin * px + cos * py + 0.01 * torch.randn_like(py)) * seen
    return x


def predict(model, data, batch=512):
    """Fall probability for each window."""
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(data), batch):
            xb = torch.as_tensor(data[i:i + batch]).to(device)
            out.append(torch.softmax(model(xb), dim=1)[:, 1].cpu())
    return torch.cat(out).numpy() if out else np.zeros(0)


def scores(prob, truth, threshold=0.5):
    """precision: of the alarms, how many were falls. recall: of the falls, how many were caught."""
    hit = prob >= threshold
    tp = int((hit & (truth == 1)).sum())
    fp = int((hit & (truth == 0)).sum())
    fn = int((~hit & (truth == 1)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("training on:", device)
model = FallSTGCN().to(device)
x_train, y_train = torch.as_tensor(X["train"]), torch.as_tensor(Y["train"])
falls, normal = int(y_train.sum()), int((y_train == 0).sum())
# far more normal windows than falls - count each fall for more, or the model
# learns that answering "never a fall" is nearly always right
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, min(20.0, normal / max(falls, 1))], device=device))
optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, EPOCHS)

CONFIG = {
    "model": "FallSTGCN", "classes": ["not_fall", "fall"],
    "pose_model": "yolo11n-pose.pt", "width": WIDTH, "rate": RATE, "window": WINDOW,
    "keypoints": "COCO-17 (x, y, confidence) from YOLO, main person", "conf_min": CONF_MIN,
    "normalise": "one hip centre per window (median), divided by the largest body extent in the window",
    "trained_on": "Le2i", "seed": SEED, "epochs": EPOCHS,
}
MODEL_PATH = WORK_DIR / "fall_stgcn.pt"
check = "val" if len(Y["val"]) else "train"
best = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    order = torch.randperm(len(x_train))
    total = 0.0
    for i in range(0, len(order), 64):
        idx = order[i:i + 64]
        if len(idx) < 2:
            continue
        loss = loss_fn(model(augment(x_train[idx].to(device))), y_train[idx].to(device))
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        total += loss.item() * len(idx)
    schedule.step()
    precision, recall, f1 = scores(predict(model, X[check]), Y[check])
    note = ""
    if f1 > best:
        best = f1
        torch.save({"state_dict": model.state_dict(), "config": CONFIG}, MODEL_PATH)
        note = "  <- best so far, saved"
    print(f"epoch {epoch:2}/{EPOCHS}  loss {total / max(len(order), 1):.3f}  "
          f"{check}: precision {precision:.2f} recall {recall:.2f} f1 {f1:.2f}{note}")

In [ ]:
# --- 5. Test on videos the model never saw ----------------------------------
model.load_state_dict(torch.load(MODEL_PATH, map_location=device)["state_dict"])

precision, recall, f1 = scores(predict(model, X["test"]), Y["test"])
print(f"test windows: precision {precision:.2f}  recall {recall:.2f}  f1 {f1:.2f}\n")

# What matters live: per video, is the fall caught, and how many alarms go off
# where there was no fall. Neighbouring windows over the line are one alarm.
test_items = [it for it in items if it["split"] == "test"]
per_video = []
for it in test_items:
    starts, samples = [], []
    for start, sample, _ in windows(it):
        starts.append(start)
        samples.append(sample)
    prob = predict(model, np.stack(samples)) if samples else np.zeros(0)
    per_video.append((it, starts, prob))

minutes = sum(len(it["kp"]) for it in test_items) / RATE / 60
results = {"test_windows": {"precision": precision, "recall": recall, "f1": f1}, "minutes_of_video": minutes}
for threshold in (0.5, 0.7, 0.9):
    caught = total_falls = false_alarms = 0
    for it, starts, prob in per_video:
        span = fall_span(it)
        total_falls += span is not None
        found = False
        k = 0
        while k < len(prob):
            if prob[k] < threshold:
                k += 1
                continue
            j = k
            while j + 1 < len(prob) and prob[j + 1] >= threshold and starts[j + 1] - starts[j] <= STRIDE:
                j += 1
            first, last = starts[k], starts[j] + WINDOW
            if span is not None and first <= span[1] + RATE and last >= span[0] - RATE:
                found = True
            else:
                false_alarms += 1
            k = j + 1
        caught += found
    results[f"threshold_{threshold}"] = {"falls_caught": caught, "falls": total_falls, "false_alarms": false_alarms}
    print(f"threshold {threshold}: caught {caught}/{total_falls} falls, "
          f"{false_alarms} false alarms in {minutes:.1f} min of test video")

(WORK_DIR / "fall_results.json").write_text(json.dumps(results, indent=2))
print("\nmodel saved:", MODEL_PATH)

## Download

Open the notebook's **Output** tab, download **`fall_stgcn.pt`**, and put it in the project's `models/` folder.
`fall_results.json` has the test numbers above.